# The parser function

The parser will take the raw JSON files and produce dictionaries, rows, clean data, .csv files of information that we need. Then, the data will be saved in the data/processed folder

In [1]:
import os
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

In [2]:
import json
import requests

from config import STATIONS
from datetime import datetime, timedelta, timezone
from pprint import pprint

import csv
import pandas as pd


In [3]:
from config import RAW_DATA_DIR, PROCESSED_DATA_DIR

Let's read in one of the .json files and print it:

In [ ]:
with open(f'{RAW_DATA_DIR}/genoa_principe_departures_2026_05_13T09_13_to_2026_05_13T17_13.json', 'r') as f:
    data = json.load(f)
    pprint(data)

{'rides': [{'calls': [{'arrival': None,
                       'departure': {'deviation': None,
                                     'platform': None,
                                     'scheduled': '2026-05-12T18:45:00Z'},
                       'sequence': 1,
                       'stop': {'city': {'id': '40df577e-8646-11e6-9066-549f350fcb0c',
                                         'name': 'Bordeaux'},
                                'code': 'BDX',
                                'description': 'Paludate bus station is '
                                               'located at Quai de Paludate, '
                                               'between the SNCF railway '
                                               'bridge and the MetPark '
                                               'Paludate parking.',
                                'id': 'dcc06cc2-9603-11e6-9066-549f350fcb0c',
                                'location': {'latitude': 44.82975,
                          

The most important arrays are:
1) data['rides'][n]['id']  
2) data['rides'][n]['status']. \

From there, data['rides'][n]['status']['deviation']['deviation_seconds'] is important for the deviation. \
General information are available at data['rides'][n]['line'] (for example 'direction' or 'code' or 'name' given the initial stop - final stop info). Here, n is the number of _rides_ that are in the .json file. In principle it's okay when data overlap. We need anyway the information of the deviations for different timestamps and stuff. 

In [31]:
data['rides'][2]['line']

{'code': '798',
 'direction': 'Marseille (Saint-Charles)',
 'name': 'Marseille - Milan',
 'trip_number': 1,
 'means_of_transport': 'BUS',
 'brand': {'id': 'a18f138c-68fa-4b45-a42f-adb0378e10d3', 'name': 'FlixBus'}}

In [32]:
print(len(data['rides']))

18


In [33]:
data['rides'][0]['status']

{'segment': None,
 'next_stop_sequence': None,
 'has_arrived_at_next_stop': True,
 'progress': None,
 'deviation': {'deviation_timestamp': '2026-05-13T09:55:17Z',
  'deviation_seconds': 3617,
  'reason': None,
  'deviation_class': 'LATE',
  'deviation_type': 'ACTUAL',
  'updated_at': '2026-05-13T09:58:33Z'},
 'scheduled_timestamp': '2026-05-13T08:55:00Z'}

In [4]:
def extract_dep_delay_id(data, n): # Extract data for the ID of the bus, general information, delay (if given), when it was updated, and the scheduled departure time

    subdata1 = data['rides'][n]['line'] # This is the general information about the bus, such as the code, direction and name of the line
    subdata2 = data['rides'][n]['status']['deviation'] if data['rides'][n]['status']['deviation'] is not None else {'deviation_seconds': None, 'updated_at': None} # For some buses there is no information about the delay


    dict_data = {
        'id': data['rides'][n]['id'],
        'bus_code': subdata1['code'],
        'direction': subdata1['direction'],
        'name': subdata1['name'],
        'deviation_seconds': subdata2['deviation_seconds'],
        'updated_at': subdata2['updated_at'],
        'scheduled_timestamp': data['rides'][n]['status']['scheduled_timestamp']
        }
    
 
    return dict_data

In [5]:
def extract_dep_delay_id_total(data): # Extract data for all the rides in the JSON file

    list_of_dicts = []

    for i in range(len(data['rides'])):
        list_of_dicts.append(extract_dep_delay_id(data, i))

    return list_of_dicts

In [113]:
extract_dep_delay_id(data, 3)

{'id': 'ea707c63-d4b0-4e14-86a9-81469eddfb56',
 'bus_code': '482',
 'direction': 'Montpellier (Sabines)',
 'name': 'Venice - Montpellier',
 'deviation_seconds': 1220,
 'updated_at': '2026-05-13T10:53:34Z',
 'scheduled_timestamp': '2026-05-13T10:30:00Z'}

## A slightly better design of the code

Chat suggests to go through the JSON files first, extracting all the data, and keep these in a big row. Then, put this big as a DataFrame. Okay. 
Not to do like a DataFrame for each of the JSON files, and then appending them to the final dataframe.

In [13]:
file_exists = Path(f'{PROCESSED_DATA_DIR}/genoa_principe_departures.csv').exists() # Boolean to check if the file already exists, if not we will create it and write the header, otherwise we will append to it without writing the header


all_rows = []

json_files_genoa = sorted(RAW_DATA_DIR.glob('genoa_principe_departures_*.json')) # Get all the JSON files for the departures from Genoa Principe, sorted by name (which includes the timestamp)

for file in json_files_genoa:

    with open(file, 'r') as f:
        data = json.load(f)
        #pprint(data) # just for debugging
    
    all_rows.extend(extract_dep_delay_id_total(data)) # Extend the list of all rows with the data extracted from the current JSON file

    # Once the looping of the files is finished

df = pd.DataFrame(all_rows) # Create a DataFrame from the list of all rows
df.to_csv(
    f'{PROCESSED_DATA_DIR}/genoa_principe_departures.csv',
    index=False
)

Btw, we will rebuild the CSV from scratch with addition to new files. It doesn't cost effectively any time, so we can do it. Otherwise, we could track which .json files we added, then have in df.to_csv(mode = 'a') the append mode on. Also, header=not file_exists can help in that case.


In [14]:
df

,id,bus_code,direction,name,deviation_seconds,updated_at,scheduled_timestamp
0,74472164-7e17-4d6e-8b42-ba3e2a2fedd1,503,Rome Tiburtina Bus station,Grenoble - Turin - Rome,1266.0,2026-05-15T13:16:41Z,2026-05-15T14:10:00Z
1,af11c615-9a63-4cb4-92a6-8012e2ba7916,N456,"Lublin, Bus Station Lublin",Lublin - Milan - Marseille,3224.0,2026-05-15T13:14:12Z,2026-05-15T14:15:00Z
2,4968f25a-1b6f-438c-a47e-ca5ce3001615,482,Venice (Tronchetto),Venice - Montpellier,408.0,2026-05-15T13:15:51Z,2026-05-15T14:30:00Z
3,d44e7e21-b17a-43e5-aa99-7205c4431bf2,500,Milan (Lampugnano bus station),Milan - Genoa/Rome/Aprilia,521.0,2026-05-15T13:08:48Z,2026-05-15T14:40:00Z
4,a18a748b-ccae-4342-a648-36a5c5c49f12,516,Naples (FS Park Stazione Centrale),Naples - Lyon,0.0,2026-05-15T13:11:09Z,2026-05-15T14:55:00Z
5,1ad052c7-6ecd-4164-a9c9-4e34094c1fc1,492,Nice Airport (Bus Station - Terminal 1),Munich - Milan- Genoa - Nice Airport ✈,0.0,2026-05-15T13:11:30Z,2026-05-15T15:15:00Z
6,056547c9-af33-4490-83e1-c471c43d597b,503,Grenoble (Central Bus Station),Grenoble - Turin - Rome,163.0,2026-05-15T13:17:17Z,2026-05-15T15:35:00Z
7,03222d3c-dcc9-4785-aad0-59e57ba886fe,1501,Florence (Villa Costanza),Florence - Dijon,0.0,2026-05-15T07:07:15Z,2026-05-15T16:20:00Z
8,cbf3475b-7f28-4194-a4e8-f49cae120bda,N718,Bordeaux (Saint-Jean - Paludate),Bordeaux - Nice - Florence,0.0,2026-05-15T12:56:06Z,2026-05-15T16:35:00Z
9,b433d016-a6ac-4568-ac11-01524dbb150e,N1501,Sala Consilina,Sala Consilina - Nice Airport,NaN,NaN,2026-05-15T17:10:00Z


In [16]:
df.head()

,id,bus_code,direction,name,deviation_seconds,updated_at,scheduled_timestamp
0,74472164-7e17-4d6e-8b42-ba3e2a2fedd1,503,Rome Tiburtina Bus station,Grenoble - Turin - Rome,1266.0,2026-05-15T13:16:41Z,2026-05-15T14:10:00Z
1,af11c615-9a63-4cb4-92a6-8012e2ba7916,N456,"Lublin, Bus Station Lublin",Lublin - Milan - Marseille,3224.0,2026-05-15T13:14:12Z,2026-05-15T14:15:00Z
2,4968f25a-1b6f-438c-a47e-ca5ce3001615,482,Venice (Tronchetto),Venice - Montpellier,408.0,2026-05-15T13:15:51Z,2026-05-15T14:30:00Z
3,d44e7e21-b17a-43e5-aa99-7205c4431bf2,500,Milan (Lampugnano bus station),Milan - Genoa/Rome/Aprilia,521.0,2026-05-15T13:08:48Z,2026-05-15T14:40:00Z
4,a18a748b-ccae-4342-a648-36a5c5c49f12,516,Naples (FS Park Stazione Centrale),Naples - Lyon,0.0,2026-05-15T13:11:09Z,2026-05-15T14:55:00Z
